# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Classification (within the Ranking Signal Analysis lane)**

My lane frames the question "which signals are associated with performance?" as a classification problem: using observed content signals as features to classify pages as declining or not (trend_direction == "down"). This is a proxy classification, not a future-prediction task yet — it lets me measure which signals actually separate declining from stable pages today.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy):** `is_declining = (trend_direction == "down")`

This is a PROXY label, not a causal or future outcome — it's derived from `trend_pct`/`trend_direction`, which are themselves computed from the current 90-day window, not a later observed outcome. Per the flyrank-data skill's label trap warning, `trend_direction` and `trend_pct` will be used ONLY as the target, never as features.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@K and effect sizes**

Since the output feeds a ranked review list for editors, precision@K (e.g., precision@50) matches how the list will actually be used — of the top K pages flagged, how many are truly declining. I will also report effect sizes for individual signals (e.g., difference in mean word_count between declining vs. stable pages) to support the signal-analysis side of this lane.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis:** one row = one pseudonymized content item (page), belonging to one client, with its 90-day trailing metrics. `content_id` and `client_id` are pseudonymous join/group keys only — never features. Note row 3 above shows a NaN in `word_count`, confirming the flyrank-data skill's warning that missingness follows content_type — this will need has_-flags rather than a blind fillna(0).

In [6]:
!git clone https://github.com/YomnaImad07/FlyRank-ML-Internship

import pandas as pd

df = pd.read_csv("/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one content item (page) for one client
unit_view = df[["content_id", "client_id", "word_count", "content_age_days",
                "avg_position", "impressions_90d", "trend_direction"]].head(5)
print(unit_view)

fatal: destination path 'FlyRank-ML-Internship' already exists and is not an empty directory.
             content_id          client_id  word_count  content_age_days  \
0  content_304f48230142  client_f369cb89fc      3221.0               187   
1  content_a1fb4e703a9e  client_4e07408562      2481.0               445   
2  content_9aa793d4d895  client_7f2253d7e2      3515.0               141   
3  content_331d6c4de07b  client_19581e27de         NaN               463   
4  content_d99b7a2d90ca  client_3fdba35f04      2803.0               263   

   avg_position  impressions_90d trend_direction  
0          10.6             3803            down  
1          20.3            15320            down  
2          36.5            12581            down  
3           6.2            11751          stable  
4          44.0            19140            down  


In [7]:
print("Rows with missing word_count:", df["word_count"].isna().sum())
print("Missing word_count by content_type (top 5):")
print(df.groupby("content_type")["word_count"].apply(lambda x: x.isna().mean()).sort_values(ascending=False).head())

Rows with missing word_count: 7699
Missing word_count by content_type (top 5):
content_type
keyword article       0.282979
comparison article    0.000000
feedly article        0.000000
Name: word_count, dtype: float64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement (e.g., "flag pages with word_count < 500") would miss interactions between signals — a low word count page with high impressions and strong position may not need review, while a page with moderate word count but sharply declining CTR might. With 30,000 pages across 32 clients and many overlapping signals (age, freshness, position, CTR, engagement), a simple rule can't weigh them together the way a model or grouped analysis can. This is exactly the "messy but real pattern" case described in framing-ml-problems.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.